In [93]:
import os
import torch
import numpy as np
import cv2
from tqdm import tqdm
from glob import glob
from PIL import Image
from pymilvus import MilvusClient
import requests
import base64
import torch
import re
import openai


In [ ]:


openai.api_key = 

In [ ]:
import os
os.environ["OPENAI_API_KEY"] =  


In [2]:
!git clone https://github.com/FlagOpen/FlagEmbedding.git
%cd FlagEmbedding/research/visual_bge
!pip install -e .

C:\Users\ilyas\FlagEmbedding\research\visual_bge


Cloning into 'FlagEmbedding'...


Obtaining file:///C:/Users/ilyas/FlagEmbedding/research/visual_bge
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for visual_bge


  DEPRECATION: Legacy editable install of visual_bge==0.1.0 from file:///C:/Users/ilyas/FlagEmbedding/research/visual_bge (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457


In [70]:
%cd .. #Execution 3 fois

C:\Users\ilyas


Note: you may need to restart the kernel to use updated packages.


In [20]:
import torch
from visual_bge.modeling import Visualized_BGE

In [22]:

class Encoder:
    def __init__(self, model_name: str, model_path: str):
        self.model = Visualized_BGE(model_name_bge=model_name, model_weight=model_path)
        self.model.eval()

    def encode_query(self, image_path: str = None, text: str = None) -> list[float]:
        with torch.no_grad():
            query_emb = self.model.encode(image=image_path, text=text)
        return query_emb.tolist()[0]

    def encode_image(self, image_path: str) -> list[float]:
        return self.encode_query(image_path=image_path)

    def encode_text(self, text: str) -> list[float]:
        return self.encode_query(text=text)


In [24]:
model_name = "BAAI/bge-base-en-v1.5"
model_path = "C:\\Users\\ilyas\\Visualized_base_en_v1.5.pth"  # Change to your own value if using a different model path
encoder = Encoder(model_name, model_path)


C:\Users\ilyas\FlagEmbedding\research\visual_bge\visual_bge\modeling.py:106: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_weight, map_

In [ ]:
# Specify your data directory
data_dir = "./images_folder"  # Update with your data directory

# Get a list of image file paths
image_list = glob(os.path.join(data_dir, "images", "*.jpg"))  # Adjust the pattern as needed

# Dictionary to store image embeddings
image_dict = {}

# Generate embeddings for each image
for image_path in tqdm(image_list, desc="Generating image embeddings: "):
    try:
        image_dict[image_path] = encoder.encode_image(image_path)
    except Exception as e:
        print(f"Failed to generate embedding for {image_path}. Skipped.")
        continue

print("Number of encoded images:", len(image_dict))


In [ ]:
dim = len(list(image_dict.values())[0])
collection_name = "multimodal_rag_demo"

# Connect to Milvus client
milvus_client = MilvusClient(uri="http://localhost:19530")

# Create Milvus Collection
milvus_client.create_collection(
    collection_name=collection_name,
    auto_id=True,
    dimension=dim,
    enable_dynamic_field=True,
)

# Insert data into the collection
milvus_client.insert(
    collection_name=collection_name,
    data=[{"image_path": k, "vector": v} for k, v in image_dict.items()],
)


In [26]:
milvus_client = MilvusClient(uri="http://localhost:19530")

DEBUG:pymilvus.milvus_client.milvus_client:Created new connection using: a7495c9047de42b3adc1b9787137b1e6


In [28]:
# Define your query parameters
query_image = None
query_text = "phone case with owl image theme and with black color"  # Set to None if not using text

# Determine the query type and generate the embedding
if query_image and query_text:
    query_vec = encoder.encode_query(image_path=query_image, text=query_text)
    query_type = "image-text"
elif query_image:
    query_vec = encoder.encode_image(image_path=query_image)
    query_type = "image-only"
elif query_text:
    query_vec = encoder.encode_text(text=query_text)
    query_type = "text-only"
else:
    raise ValueError("At least one of query_image or query_text must be provided.")


In [32]:
collection_name = "multimodal_rag_demo"

In [34]:
# Perform the search in Milvus
search_results = milvus_client.search(
    collection_name=collection_name,
    data=[query_vec],
    output_fields=["image_path"],
    limit=9,  # Adjust as needed
    search_params={"metric_type": "COSINE", "params": {}},
)[0]

# Retrieve image paths from the search results
retrieved_images = [hit.get("entity").get("image_path") for hit in search_results]


In [48]:
retrieved_images

['./images_folder\\images\\41n00AOfWhL._AC_.jpg',
 './images_folder\\images\\516PebbMAcL._AC_.jpg',
 './images_folder\\images\\51rOcop42NL._AC_.jpg',
 './images_folder\\images\\51x4++eFD0L._AC_.jpg',
 './images_folder\\images\\51TnFCnglcL._AC_.jpg',
 './images_folder\\images\\518Gj1WQ-RL._AC_.jpg',
 './images_folder\\images\\51Azb8eaMfL._AC_.jpg',
 './images_folder\\images\\41uX8POiX9L._AC_.jpg',
 './images_folder\\images\\41Mk8fIjDRL._AC_.jpg']

In [74]:
def create_panoramic_view(query_image_path: str, retrieved_images: list) -> np.ndarray:
    """
    Creates a panoramic view combining the query image and retrieved images.

    Args:
        query_image_path (str): Path to the query image. Can be None for text-only queries.
        retrieved_images (list): List of paths to retrieved images.

    Returns:
        np.ndarray: The combined panoramic image.
    """
    # Parameters for image sizes and layout
    img_height = 300
    img_width = 300
    row_count = 3  # Number of images per row
    border_size = 10  # Border size for the query image

    # Calculate panoramic image dimensions
    panoramic_width = img_width * row_count
    panoramic_height = img_height * row_count

    # Initialize a white background for the panoramic image
    panoramic_image = np.full(
        (panoramic_height, panoramic_width, 3), 255, dtype=np.uint8
    )

    # Create a placeholder for the query image or text
    query_image_placeholder = np.full(
        (panoramic_height, img_width, 3), 255, dtype=np.uint8
    )

    # Handle the query image
    if query_image_path:
        # Load and resize the query image
        query_image = Image.open(query_image_path).convert("RGB")
        query_array = np.array(query_image)[:, :, ::-1]  # Convert RGB to BGR for OpenCV
        resized_query_image = cv2.resize(query_array, (img_width, img_height))

        # Add a blue border to the query image
        blue = (255, 0, 0)  # Blue color in BGR
        bordered_query_image = cv2.copyMakeBorder(
            resized_query_image,
            border_size,
            border_size,
            border_size,
            border_size,
            cv2.BORDER_CONSTANT,
            value=blue,
        )

        # Place the query image in the placeholder
        query_image_placeholder[
            img_height * 2 : img_height * 3, 0:img_width
        ] = cv2.resize(bordered_query_image, (img_width, img_height))

        # Add text "Query Image" below the query image
        cv2.putText(
            query_image_placeholder,
            "Query Image",
            (10, img_height * 3 + 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            blue,
            2,
            cv2.LINE_AA,
        )
    else:
        # For text-only queries, display the query text
        cv2.putText(
            query_image_placeholder,
            "Text Query",
            (10, img_height * 3 + 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 0),
            2,
            cv2.LINE_AA,
        )

    # Process retrieved images
    for i, image_path in enumerate(retrieved_images):
        try:
            # Load and resize the retrieved image
            print(image_path)
            print("aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq")
            retrieved_image = Image.open(image_path).convert("RGB")
            print("aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq")
            retrieved_array = np.array(retrieved_image)[:, :, ::-1]
            resized_image = cv2.resize(retrieved_array, (img_width - 4, img_height - 4))
            print("aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq")
            # Add a black border to the retrieved image
            bordered_image = cv2.copyMakeBorder(
                resized_image,
                2,
                2,
                2,
                2,
                cv2.BORDER_CONSTANT,
                value=(0, 0, 0),
            )

            # Calculate the position of the image in the panoramic view
            row = i // row_count
            col = i % row_count
            start_row = row * img_height
            start_col = col * img_width

            # Place the retrieved image in the panoramic image
            panoramic_image[
                start_row : start_row + img_height, start_col : start_col + img_width
            ] = bordered_image

            # Add red index numbers to each image
            index_text = str(i)
            cv2.putText(
                panoramic_image,
                index_text,
                (start_col + 10, start_row + 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2,
                cv2.LINE_AA,
            )
        except Exception as e:
            print(f"Error processing image {image_path}: {e}")

    # Combine the query placeholder with the panoramic images
    combined_panoramic = np.hstack([query_image_placeholder, panoramic_image])

    return combined_panoramic


In [76]:
data_dir ="C:\\Users\\ilyas\\images_folder"

In [78]:
# Create the panoramic view
panoramic_image = create_panoramic_view(query_image, retrieved_images)

# Save the panoramic image
combined_image_path = os.path.join(data_dir, "combined_image.jpg")
cv2.imwrite(combined_image_path, panoramic_image)

# Display the panoramic image
combined_image = Image.open(combined_image_path)
combined_image.show()


./images_folder\images\41n00AOfWhL._AC_.jpg
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
./images_folder\images\516PebbMAcL._AC_.jpg
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
./images_folder\images\51rOcop42NL._AC_.jpg
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
./images_folder\images\51x4++eFD0L._AC_.jpg
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
./images_folder\images\51TnFCnglcL._AC_.jpg
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
aqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqqq
./images_folder\images\518Gj1W

In [109]:
import requests
import base64
import re

def generate_ranking_explanation(
    combined_image_path: str, caption: str, query_type: str, infos: dict = None
) -> tuple[list[int], str]:
    """
    Generates a ranking explanation using an LLM.

    Args:
        combined_image_path (str): Path to the combined panoramic image.
        caption (str): The query text.
        query_type (str): Type of the query ('image-text', 'image-only', 'text-only').
        infos (dict, optional): Additional information about the images.

    Returns:
        tuple[list[int], str]: Ranked indices and the explanation text.
    """
    with open(combined_image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    # Build the prompt for the LLM
    information = (
        "You are responsible for ranking results for a Retrieval task. "
        "The user provides an input indicating their retrieval intent. "
    )

    # Adjust the prompt based on the query type
    if query_type == "image-text":
        information += (
            "The user provides both an image and an instruction. "
            f"User instruction: {caption}\n\n"
        )
    elif query_type == "image-only":
        information += "The user provides an image indicating their retrieval intent.\n\n"
    elif query_type == "text-only":
        information += (
            "The user provides an instruction indicating their retrieval intent. "
            f"User instruction: {caption}\n\n"
        )
    else:
        raise ValueError("Invalid query type specified.")

    information += (
        "You will receive the query (with a blue border) and the retrieved images. "
        "Each item has its red index number in the top left corner. Do not confuse the index numbers with the content. "
    )

    # Include additional information if provided
    if infos:
        for i, info in enumerate(infos.get("product", [])):
            information += f"{i}. {info}\n"

    information += (
        "Provide a new ranked list of indices from most suitable to least suitable, "
        "followed by an explanation for the top 2 most suitable items only. "
        "The format of the response must be 'Ranked list: [indices]' with the indices as integers, "
        "followed by 'Reasons:' and the explanation."
    )

    # Prepare the payload for the OpenAI API
    payload = {
        "model": "gpt-4o",  # Use the updated model name
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": information,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        },
                    },
                ],
            }
        ],
        "max_tokens": 300,
    }

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {openai_api_key}",
    }

    response = requests.post(
        "https://api.openai.com/v1/chat/completions", headers=headers, json=payload
    )

    # Check for errors in the response
    if response.status_code != 200:
        print(f"Request failed with status code {response.status_code}: {response.text}")
        raise Exception(f"OpenAI API request failed with status code {response.status_code}")

    result = response.json()["choices"][0]["message"]["content"]

    # Parse the ranked indices from the response
    match = re.search(r"\[([0-9,\s]+)\]", result)
    if match:
        ranked_indices_str = match.group(1).split(",")
        ranked_indices = [int(index.strip()) for index in ranked_indices_str]
    else:
        ranked_indices = []

    # Extract explanation
    if "Reasons:" in result:
        explanation = result.split("Reasons:", 1)[-1].strip()
    else:
        explanation = ""

    return ranked_indices, explanation


In [111]:
# Generate ranking explanation using the LLM
ranked_indices, explanation = generate_ranking_explanation(
    combined_image_path, query_text or "", query_type
)



In [113]:
print("Explanation from LLM:")
print(explanation)

Explanation from LLM:
1. **Index 0**: This phone case features an image of an owl with a prominent black color, matching the user’s intent perfectly.
2. **Index 4**: This case also has an owl-themed design. Although it incorporates more colors, black is present in the design, making it suitable for the user’s intent.


In [115]:
# Display the best matched image based on LLM ranking
best_index = ranked_indices[0]
best_image_path = retrieved_images[best_index]
best_image = Image.open(best_image_path)
best_image = best_image.resize((150, 150))
best_image.show()
